In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoAlertPresentException
from datetime import datetime
import time
import traceback

TARGET_URL     = "https://isrc.snu.ac.kr/hm/route/progress/schedule/view?sorderSkey=13277"
PROCESS_NAME     = "ICP Metal Etcher (03)"
START_DATETIME = "2026-07-09 11:00"
END_DATETIME   = "2026-07-09 13:00"

def dismiss_alert(driver):
    try:
        alert = driver.switch_to.alert
        print(f"  [알림 닫음: '{alert.text}']")
        alert.accept()
        time.sleep(0.3)
        return True
    except NoAlertPresentException:
        return False

def js_set(driver, element, value):
    driver.execute_script("""
        var el = arguments[0], val = arguments[1];
        el.value = val;
        el.dispatchEvent(new Event('input',  {bubbles:true}));
        el.dispatchEvent(new Event('change', {bubbles:true}));
    """, element, value)

def find_예약_btn(driver):
    candidates = [
        el for el in driver.find_elements(By.XPATH,
            "//button[contains(.,'예약') and not(contains(.,'신청'))]"
            " | //a[contains(.,'예약') and not(contains(.,'신청'))]")
        if el.is_displayed()
    ]
    return candidates[-1] if candidates else None

def click_ok_dialog(driver, timeout=8):
    for i in range(timeout * 5):
        time.sleep(0.2)
        try:
            driver.switch_to.default_content()
        except Exception:
            pass

        confirm_els = [el for el in driver.find_elements(By.XPATH,
            "//*[contains(text(),'예약하시겠습니까')]")
            if el.is_displayed()]
        if not confirm_els:
            continue

        print(f"  '예약하시겠습니까?' 감지 (시도#{i+1})")

        ok_btn = driver.execute_script("""
            var walker = document.createTreeWalker(
                document.body, NodeFilter.SHOW_TEXT, null, false);
            var node;
            while (node = walker.nextNode()) {
                if (node.textContent.includes('예약하시겠습니까')) {
                    var p = node.parentElement;
                    for (var i = 0; i < 10; i++) {
                        if (!p || p.tagName === 'BODY') break;
                        var btns = p.querySelectorAll('button, a');
                        for (var btn of btns) {
                            var t = btn.textContent.trim();
                            if (t === 'Ok' || t === 'OK') return btn;
                        }
                        for (var btn of btns) {
                            if (btn.textContent.trim() === '확인') return btn;
                        }
                        p = p.parentElement;
                    }
                }
            }
            return null;
        """)

        if ok_btn:
            tag = ok_btn.tag_name
            txt = driver.execute_script("return arguments[0].textContent.trim();", ok_btn)
            print(f"  Ok버튼 (다이얼로그 내): <{tag}> '{txt}'")
            try:
                ok_btn.click(); print("  → native click")
            except Exception:
                pass
            time.sleep(0.15)
            try:
                ActionChains(driver).move_to_element(ok_btn).click().perform()
                print("  → ActionChains")
            except Exception:
                pass
            time.sleep(0.15)
            try:
                ok_btn.send_keys(Keys.RETURN); print("  → send_keys ENTER")
            except Exception:
                pass
            time.sleep(0.5)
            return True

        print("  ⚠️ 다이얼로그 내 Ok버튼 못 찾음")

    print("  ⚠️ '예약하시겠습니까?' 미감지")
    return dismiss_alert(driver)

# ─────────────────────────────────────────────
def _read_time_js(driver):
    """현재 컨텍스트에서 시각 추출 (한국어 '17시 49분 51초' 형식)"""
    return driver.execute_script(r"""
        var body = document.body.innerText || document.body.textContent || '';
        // "17시 49분 51초" 형식 (navyism)
        var m = body.match(/(\d{1,2})시\s*(\d{1,2})분\s*(\d{1,2})초/);
        if (m) return m[1] + ':' + m[2].padStart(2,'0') + ':' + m[3].padStart(2,'0');
        // "HH:MM:SS" 형식 (fallback)
        var m2 = body.match(/(\d{1,2}:\d{2}:\d{2})/);
        return m2 ? m2[1] : null;
    """)

def get_navyism_time(driver):
    """navyism.com 새 탭에서 시각 읽기 (최대 8초 폴링)"""
    original = driver.current_window_handle
    try:
        driver.execute_script("window.open('https://time.navyism.com/');")
        time.sleep(2)
        new_win = [w for w in driver.window_handles if w != original][-1]
        driver.switch_to.window(new_win)

        result = None
        for _ in range(16):   # 0.5s × 16 = 최대 8초
            time.sleep(0.5)
            result = _read_time_js(driver)
            if result:
                break

        driver.close()
        driver.switch_to.window(original)

        if result:
            parts = result.split(':')
            h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
            print(f"  [navyism 시각: {h:02d}:{m:02d}:{s:02d}]")
            return h, m, s

    except Exception as e:
        print(f"  [navyism 오류: {e}]")
        try:
            driver.close()
            driver.switch_to.window(original)
        except Exception:
            pass
    return None

def wait_until_midnight(driver):
    """navyism 기준 자정(00:00:00)까지 대기"""
    print("\n⏳ navyism 시각 확인 중...")

    while True:
        t = get_navyism_time(driver)
        if t is None:
            print("  navyism 읽기 실패 → 시스템 시간 사용")
            now = datetime.now()
            t = (now.hour, now.minute, now.second)

        h, m, s = t
        total_secs = h * 3600 + m * 60 + s
        remaining = (24 * 3600) - total_secs

        if remaining >= 24 * 3600 or remaining == 0:
            print("  ⏰ 자정 도달! 예약 시작")
            return

        print(f"  현재 {h:02d}:{m:02d}:{s:02d} → 자정까지 {remaining//3600}시간 {(remaining%3600)//60}분 {remaining%60}초")

        if remaining > 30:
            sleep_secs = remaining - 25
            print(f"  {sleep_secs//60}분 {sleep_secs%60}초 후 재확인...")
            time.sleep(sleep_secs)
        else:
            print("  자정 30초 이내 — 초 단위 대기 시작...")
            while True:
                now = datetime.now()
                if now.hour == 0 and now.minute == 0 and now.second == 0:
                    print("  ⏰ 자정 도달!")
                    return
                time.sleep(0.05)

# ─────────────────────────────────────────────

def run_booking():
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 15)

    try:
        print("STEP 1: 페이지 이동...")
        driver.get(TARGET_URL)
        time.sleep(1)
        print(f"  URL: {driver.current_url}")
        if "login" in driver.current_url.lower():
            print("  ⚠️ 로그인 필요 → 브라우저에서 로그인 후 Enter")
            input("  로그인 완료 후 Enter: ")
            driver.get(TARGET_URL)
            time.sleep(1)

        wait_until_midnight(driver)

        print("STEP 2-5: 로드 → 드롭다운 → 입력 → 검색...")
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
        time.sleep(1)
        Select(driver.find_element(By.NAME, "searchType")).select_by_value("route")
        time.sleep(0.3)
        inp = driver.find_element(By.NAME, "typeValue")
        inp.clear()
        inp.send_keys(PROCESS_NAME)
        driver.find_element(By.XPATH, "//a[contains(text(),'검색')]").click()
        time.sleep(1)

        print("STEP 6: 장비예약 클릭...")
        a_list = [
            el for el in driver.find_elements(By.XPATH, "//a[normalize-space(.)='장비예약']")
            if not (el.get_attribute("href") or "").startswith("http")
        ]
        btn_list = driver.find_elements(By.XPATH, "//button[normalize-space(.)='장비예약']")
        target = (btn_list or a_list)[0]
        driver.execute_script("arguments[0].scrollIntoView(true);", target)
        time.sleep(0.3)
        driver.execute_script("arguments[0].click();", target)
        time.sleep(1)

        print("STEP 7: 날짜/시간 직접 입력...")
        fmt = "%Y-%m-%d %H:%M"
        use_minutes = int((datetime.strptime(END_DATETIME, fmt)
                         - datetime.strptime(START_DATETIME, fmt)).total_seconds() / 60)
        js_set(driver, driver.find_element(By.NAME, "START_DT_STR"), START_DATETIME)
        print(f"  START_DT_STR = {START_DATETIME}")
        time.sleep(0.2); dismiss_alert(driver)
        js_set(driver, driver.find_element(By.NAME, "END_DT_STR"), END_DATETIME)
        print(f"  END_DT_STR   = {END_DATETIME}")
        time.sleep(0.2); dismiss_alert(driver)
        js_set(driver, driver.find_element(By.NAME, "USE_TIME"), str(use_minutes))
        print(f"  USE_TIME     = {use_minutes}분")
        time.sleep(0.3); dismiss_alert(driver)

        print("STEP 8: 예약신청 클릭...")
        original_window = driver.current_window_handle
        btn = driver.find_element(By.XPATH,
            "//a[contains(.,'예약신청')] | //button[contains(.,'예약신청')]")
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.5); dismiss_alert(driver)

        print("STEP 9: 공정 예약 팝업 → 예약 버튼...")
        예약_btn = None
        if len(driver.window_handles) > 1:
            for wh in driver.window_handles:
                if wh != original_window:
                    driver.switch_to.window(wh)
                    print("  새 창 전환됨"); break
            예약_btn = find_예약_btn(driver)

        if 예약_btn is None:
            driver.switch_to.default_content()
            for idx, iframe in enumerate(driver.find_elements(By.TAG_NAME, "iframe")):
                try:
                    driver.switch_to.default_content()
                    driver.switch_to.frame(iframe)
                    b = find_예약_btn(driver)
                    if b:
                        예약_btn = b
                        print(f"  iframe[{idx}]에서 발견"); break
                except Exception:
                    pass

        if 예약_btn is None:
            driver.switch_to.default_content()
            예약_btn = find_예약_btn(driver)

        if 예약_btn is None:
            raise Exception("예약 버튼을 찾지 못함")

        try:
            예약_btn.click()
        except Exception:
            ActionChains(driver).move_to_element(예약_btn).click().perform()
        print("  예약 클릭")

        try:
            driver.switch_to.default_content()
        except Exception:
            pass
        time.sleep(1)

        print("STEP 10: 예약하시겠습니까? → Ok...")
        click_ok_dialog(driver, timeout=8)
        print("✅ 최종 예약 완료!")

    except Exception as e:
        print(f"\n❌ 오류: {type(e).__name__}: {e}")
        traceback.print_exc()
        dismiss_alert(driver)
    finally:
        input("\nEnter로 브라우저 닫기...")
        driver.quit()


run_booking()


STEP 1: 페이지 이동...
  URL: https://isrc.snu.ac.kr/login
  ⚠️ 로그인 필요 → 브라우저에서 로그인 후 Enter

⏳ navyism 시각 확인 중...
  [navyism 오류: list index out of range]
  navyism 읽기 실패 → 시스템 시간 사용
  현재 18:09:26 → 자정까지 5시간 50분 34초
  350분 9초 후 재확인...
